In [ ]:
'''
1. 문서 로드
2. 문서 split
3. embedding 설정
4. db (FAISS) 설정
5. retrieval (검색) + (리랭커)
6. 프롬프트
7. 모델
8. 결과


'''

In [1]:
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("RAG-practice")

LangSmith 추적을 시작합니다.
[프로젝트명]
RAG-practice


In [2]:
# 캐시 설정
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

set_llm_cache(SQLiteCache(database_path='./cache/rag1.db'))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
# 문서 로드
loader = PyMuPDFLoader("datas/SPRI_AI_Brief_2023년12월호_F.pdf")


문서의 페이지 수: 23


In [ ]:
# 문서 분할


분할된 청크의 수: 43


In [ ]:
# 임베딩 설정


In [ ]:
# DB 생성 및 저장
# 벡터 스토어로 저장


In [ ]:
# 검색기 생성


In [ ]:
# 프롬프트 생성




PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\n당신은 훌륭한 응답 도우미 입니다. 주어진 context를 이용해서 question에 답하세요.\n모르겠으면 <i don't know> 라고 말하세요.\n그리고 한국말로 답하세요.\n\n#Question:\n{question}\n#Context:\n{context}\n")

In [ ]:
# llm 생성


In [ ]:
# chain 생성


In [19]:
question = '삼성전자가 자체 개발한 AI의 이름은?'
response = chain.invoke(question)
print(response)

삼성전자가 자체 개발한 AI의 이름은 '삼성 가우스'입니다.


In [20]:
# 네이버 뉴스 기사이용하기
"""
똑같이
문서 로드 -> 문서 분할 -> 임베딩 설정 -> DB 설정 ->
검색기 설정 -> 프롬프트 -> 모델 -> chain(outputparser포함해서)
"""


'\n똑같이\n문서 로드 -> 문서 분할 -> 임베딩 설정 -> DB 설정 ->\n검색기 설정 -> 프롬프트 -> 모델 -> chain(outputparser포함해서)\n'

In [24]:
import bs4
from langchain_classic import hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [25]:
# bs4로 편하게 웹에서 원하는 요소를 가져오기
bs4.SoupStrainer(
    "div",
    attrs={"class": ["newsct_article _article_body", "media_end_head_title"]},
)

<SoupStrainer name=[<TagNameMatchRule string=div pattern=None function=None present=None>] attrs=defaultdict(<class 'list'>, {'class': [<AttributeValueMatchRule string=newsct_article _article_body pattern=None function=None present=None>, <AttributeValueMatchRule string=media_end_head_title pattern=None function=None present=None>]}) string=[]>

In [ ]:
# 문서 로드
loader = WebBaseLoader(
    web_paths=("https://n.news.naver.com/article/437/0000378416",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": ["newsct_article _article_body", "media_end_head_title"]},
        )
    ),
)


문서의 수: 1


[Document(metadata={'source': 'https://n.news.naver.com/article/437/0000378416'}, page_content="\n출산 직원에게 '1억원' 쏜다…회사의 파격적 저출생 정책\n\n\n[앵커]올해 아이 낳을 계획이 있는 가족이라면 솔깃할 소식입니다. 정부가 저출생 대책으로 매달 주는 부모 급여, 0세 아이는 100만원으로 올렸습니다. 여기에 첫만남이용권, 아동수당까지 더하면 아이 돌까지 1년 동안 1520만원을 받습니다. 지자체도 경쟁하듯 지원에 나섰습니다. 인천시는 새로 태어난 아기, 18살될 때까지 1억원을 주겠다. 광주시도 17살될 때까지 7400만원 주겠다고 했습니다. 선거 때면 나타나서 아이 낳으면 현금 주겠다고 밝힌 사람이 있었죠. 과거에는 표만 노린 '황당 공약'이라는 비판이 따라다녔습니다. 그런데 지금은 출산율이 이보다 더 나쁠 수 없다보니, 이런 현금성 지원을 진지하게 정책화 하는 상황까지 온 겁니다. 게다가 기업들도 뛰어들고 있습니다. 이번에는 출산한 직원에게 단번에 1억원을 주겠다는 회사까지 나타났습니다.이상화 기자가 취재했습니다.[기자]한 그룹사가 오늘 파격적인 저출생 정책을 내놨습니다.2021년 이후 태어난 직원 자녀에 1억원씩, 총 70억원을 지원하고 앞으로도 이 정책을 이어가기로 했습니다.해당 기간에 연년생과 쌍둥이 자녀가 있으면 총 2억원을 받게 됩니다.[오현석/부영그룹 직원 : 아이 키우는 데 금전적으로 많이 힘든 세상이잖아요. 교육이나 생활하는 데 큰 도움이 될 거라 생각합니다.]만약 셋째까지 낳는 경우엔 국민주택을 제공하겠다는 뜻도 밝혔습니다.[이중근/부영그룹 회장 : 3년 이내에 세 아이를 갖는 분이 나올 것이고 따라서 주택을 제공할 수 있는 계기가 될 것으로 생각하고.][조용현/부영그룹 직원 : 와이프가 셋째도 갖고 싶어 했는데 경제적 부담 때문에 부정적이었거든요. (이제) 긍정적으로 생각할 수 있을 것 같습니다.]오늘 행사에서는, 회사가 제공하는 출산장려금은 받는 

In [ ]:
# 문서 분할


분할된 문서의 수: 3


In [ ]:
# 임베딩 설정


In [ ]:
# vector store 생성


In [ ]:
# 검색기 설정


In [ ]:
# 프롬프트
from langchain_core.prompts import PromptTemplate

template = """
당신은 질문-답변(Question-Answering)을 수행하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context) 에서 주어진 질문(question) 에 답하는 것입니다.
검색된 다음 문맥(context) 을 사용하여 질문(question) 에 답하세요. 만약, 주어진 문맥(context) 에서 답을 찾을 수 없다면, 답을 모른다면 `주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다` 라고 답하세요.
한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.

#Question: 
{question} 

#Context: 
{context} 
"""



PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\n당신은 질문-답변(Question-Answering)을 수행하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context) 에서 주어진 질문(question) 에 답하는 것입니다.\n검색된 다음 문맥(context) 을 사용하여 질문(question) 에 답하세요. 만약, 주어진 문맥(context) 에서 답을 찾을 수 없다면, 답을 모른다면 `주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다` 라고 답하세요.\n한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.\n\n#Question: \n{question} \n\n#Context: \n{context} \n')

In [ ]:
# model


In [ ]:
# chain


In [35]:
question = "부영그룹의 출산 장려 정책에 대해 설명해주세요."
answer = chain.invoke(question)
answer

'부영그룹의 출산 장려 정책은 매우 파격적입니다. 이 그룹은 2021년 이후 태어난 직원 자녀에게 1억원씩 지원하기로 했으며, 총 70억원을 이 정책에 할당했습니다. 만약 연년생이나 쌍둥이 자녀가 있을 경우, 총 2억원을 받을 수 있습니다. 또한, 셋째 자녀를 낳는 경우에는 국민주택을 제공하겠다는 계획도 밝혔습니다. 이 정책은 출산에 대한 경제적 부담을 덜어주고, 직원들이 아이를 더 낳는 데 긍정적인 영향을 미칠 것으로 기대되고 있습니다.'